# Live demo: one test pair, every model

Picks a pair from the test split, runs both encoders on the raw image and title, and shows the
top-3 classes of every trained model. Nothing is read from the feature cache or the prediction
files for that pair; the last section checks that the live pass gives the same answer as the
test run.

Needs the data, the feature cache and the checkpoints under `outputs/` (the training recipes of
the README, seed 0). Register the kernel once with `just kernel`, then pick the `recipe-fusion`
kernel. The first run also computes the training means (about a minute) and caches them in
`outputs/demo/`.

In [ ]:
import time
from pathlib import Path

from IPython.display import HTML, display
from PIL import Image

from recipe_fusion import demo as D
from recipe_fusion.data.upmc import read_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEED = 0

started = time.perf_counter()
demo = D.load_demo(ROOT, seed=SEED)
test = read_split(D.Paths.from_root(ROOT).upmc_dir, "test")
models = list(demo.heads) + (["text_bert_ft"] if demo.bert_ft else []) + list(demo.late)
print(f"{len(models)} models on {demo.device} in {time.perf_counter() - started:.0f} s: {models}")

## Pick a pair

`CHOICE`: `None` for a random pair, a position in the test file (0 to 22,715), an image file
name such as `"apple_pie_123.jpg"`, or a class name such as `"sushi"` for a random pair of that
class. `TITLE` replaces the real title, to try a title of your own on the same photo.

In [ ]:
CHOICE = None
TITLE = None

sample = D.pick(test, CHOICE)
image = Image.open(sample.image_path(D.Paths.from_root(ROOT).upmc_dir)).convert("RGB")
title = TITLE if TITLE is not None else sample.text

display(image.resize((320, round(320 * image.height / image.width))))
print(f"title: {title}")
print(f"truth: {sample.label}   ({sample.image_name}, test position {test.index(sample)})")

## Every model on the pair

In [ ]:
started = time.perf_counter()
encoded = demo.encode(image, title)
logits = demo.predict(encoded)
print(f"encoded and scored in {time.perf_counter() - started:.2f} s")
display(HTML(D.to_html(logits, demo.classes, sample.label)))

## Without the title

The title is replaced by the training mean, as in the test-time ablations. The fine-tuned BERT
reads the raw text, so it and its late fusion drop out.

In [ ]:
display(
    HTML(D.to_html(demo.predict(D.remove(encoded, "text", demo.means)), demo.classes, sample.label))
)

## Without the image

In [ ]:
display(
    HTML(
        D.to_html(demo.predict(D.remove(encoded, "image", demo.means)), demo.classes, sample.label)
    )
)

## Check against the test run

Largest absolute difference between the live pass and the cached features, then the saved test
logits of the same pair; the top-1 should agree everywhere. Small differences come from bfloat16
on the GPU and batch size. Only meaningful with the real title (`TITLE = None`).

In [ ]:
for row in D.check_against_test(ROOT, sample, encoded, logits, seed=SEED):
    agree = (
        ""
        if row["same_top1"] is None
        else ("same top-1" if row["same_top1"] else "DIFFERENT top-1")
    )
    print(f"{row['what']:<22} max |diff| {row['max_abs_diff']:.4f}  {agree}")